In [1]:
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, f1_score
import shap
import numpy as np
import pandas as pd
import sys
import os
from pathlib import Path
from sklearn.model_selection import train_test_split
from statsmodels.tools.tools import add_constant
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [2]:
def find_project_root(current_path, marker="requirements.txt"):
    current = Path(current_path).resolve()
    for _ in range(5):
        if (current / marker).exists():
            return current
        current = current.parent
    raise FileNotFoundError(f"Не удалось найти корень проекта (файл {marker})")


PROJECT_ROOT = find_project_root(os.getcwd())
sys.path.append(str(PROJECT_ROOT))
print(f"Корневая директория: {PROJECT_ROOT}")

Корневая директория: D:\Education\Arcticle\02_dtp_project_JAER


# Данные

In [3]:
df = pd.read_parquet(rf"{PROJECT_ROOT}\data\processed\dtp_msk.parquet")
df

,light_cat,scheme,category,year,month_sin,month_cos,hour_sin,hour_cos,is_weekend,near_residential,...,ppl_drv_novice,viol_priority,viol_pedestrian,viol_speed,viol_drunk,viol_safety,viol_rights,viol_runaway,viol_admin,target
0,day,200,столкновение,2023,1.224647e-16,-1.000000,-0.965926,-2.588190e-01,0,1,...,1,0,1,0,0,0,0,0,0,0
1,day,300,столкновение,2015,8.660254e-01,-0.500000,-1.000000,-1.836970e-16,0,1,...,0,1,0,0,0,0,1,0,1,0
2,day,500,столкновение,2015,8.660254e-01,0.500000,-0.866025,-5.000000e-01,0,0,...,0,1,0,0,0,0,0,0,0,1
3,day,820,наезд_на_пешехода,2015,8.660254e-01,0.500000,0.707107,-7.071068e-01,0,1,...,0,0,1,0,0,0,0,0,0,0
4,day,300,столкновение,2015,8.660254e-01,0.500000,-1.000000,-1.836970e-16,0,1,...,0,1,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91744,day,950,опрокидывание,2025,1.224647e-16,-1.000000,-1.000000,-1.836970e-16,0,1,...,0,0,0,1,0,0,0,0,0,0
91745,day,070,столкновение,2025,1.224647e-16,-1.000000,0.866025,-5.000000e-01,1,0,...,0,0,0,1,0,0,0,0,0,1
91746,day,300,столкновение,2025,5.000000e-01,-0.866025,0.866025,-5.000000e-01,1,0,...,0,1,0,0,0,0,0,0,1,2
91747,night_lit,200,столкновение,2025,5.000000e-01,-0.866025,-0.258819,9.659258e-01,1,1,...,1,1,0,0,1,0,0,0,0,1


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 91749 entries, 0 to 91748
Data columns (total 89 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   light_cat                       91749 non-null  object 
 1   scheme                          91749 non-null  object 
 2   category                        91749 non-null  object 
 3   year                            91749 non-null  int32  
 4   month_sin                       91749 non-null  float64
 5   month_cos                       91749 non-null  float64
 6   hour_sin                        91749 non-null  float64
 7   hour_cos                        91749 non-null  float64
 8   is_weekend                      91749 non-null  int32  
 9   near_residential                91749 non-null  int64  
 10  near_pedestrian_infrastructure  91749 non-null  int64  
 11  near_road_junctions             91749 non-null  int64  
 12  near_education                  

In [5]:
print(df["target"].value_counts())

target
0    66704
1    20945
2     4100
Name: count, dtype: int64


In [6]:
nans = df.isna().sum()
nans = nans[nans > 0]
if not nans.empty:
    print("НАЙДЕНЫ ПРОПУСКИ (NaN):")
    print(nans)
else:
    print("Пропусков нет. Датасет плотный.")

Пропусков нет. Датасет плотный.


In [7]:
print("ТИПЫ ДАННЫХ:")
print(df.dtypes.value_counts())

ТИПЫ ДАННЫХ:
int64      71
float64    11
object      4
int32       3
Name: count, dtype: int64


In [8]:
print(df["region_id"].value_counts()[:10])

region_id
москва__выхиножулебино      1399
москва__люблино             1356
москва__пресненский         1350
москва__спецтрассы          1347
москва__южное_бутово        1323
москва__марьино             1256
москва__даниловский         1247
москва__митино              1236
москва__строгино            1204
москва__хорошевомневники    1196
Name: count, dtype: int64


In [9]:
df['target'] = (df['target'] > 0).astype(int)

In [10]:
print(df["target"].value_counts())

target
0    66704
1    25045
Name: count, dtype: int64


## Удалим утечки

In [11]:
# Честная очистка от Data Leakage (удаляем описательные переменные)
leakage_cols = [
    'category', 'scheme', 'ppl_count', 'ppl_pass_count', 'ppl_ped_count', 
    'ppl_solo_count', 'ppl_ped_worker_count', 'ppl_ped_other_count',
    'vh_count', 'vh_count_car', 'vh_count_truck', 'vh_count_bus', 
    'vh_count_moto', 'vh_count_special', 'vh_count_brand_ru', 
    'vh_count_brand_premium', 'vh_count_brand_chinese', 'vh_count_brand_mass', 
    'vh_count_brand_commercial', 'vh_count_color_dark', 'vh_count_color_light', 
    'vh_count_color_colored', 'vh_is_mass', 'nearby_objects_count'
]

# Удаляем эти колонки из датасета
df_clean = df.drop(columns=leakage_cols, errors='ignore')

In [12]:
# --- 0. БИНАРИЗАЦИЯ ТАРГЕТА ---
df_clean['target'] = (df_clean['target'] > 0).astype(int)
print(f"Баланс классов:\n{df_clean['target'].value_counts()}")

Баланс классов:
target
0    66704
1    25045
Name: count, dtype: int64


In [13]:
# --- 1. ПОДГОТОВКА ДАННЫХ ДЛЯ CATBOOST ---
X_cat = df_clean.drop(columns=['target'])
y = df_clean['target']

cat_features = [c for c in ['light_cat', 'region_id'] if c in X_cat.columns]

In [14]:
# --- 2. ПОДГОТОВКА ДАННЫХ ДЛЯ ЛОГИТА (ЭКОНОМЕТРИКИ) ---
# Линейные модели не "переварят" координаты и 147 ID районов
cols_to_drop = ['region_id', 'lat', 'long']
X_logit_raw = X_cat.drop(columns=[c for c in cols_to_drop if c in X_cat.columns])

# Делаем One-Hot Encoding (drop_first=True спасает от жесткой коллинеарности)
if 'light_cat' in X_logit_raw.columns:
    X_logit = pd.get_dummies(X_logit_raw, columns=['light_cat'], drop_first=True)
else:
    X_logit = X_logit_raw.copy()

In [15]:
# --- 3. РАЗБИЕНИЕ НА TRAIN / TEST ---
X_cat_train, X_cat_test, y_train, y_test = train_test_split(
    X_cat, y, test_size=0.2, random_state=42, stratify=y
)

X_logit_train, X_logit_test, _, _ = train_test_split(
    X_logit, y, test_size=0.2, random_state=42, stratify=y
)

In [16]:
# ==========================================
# ШАГ 4. ОБУЧЕНИЕ CATBOOST
# ==========================================
print("--- Обучение CatBoost ---")
cb_model = CatBoostClassifier(iterations=500, random_seed=42, auto_class_weights='Balanced', verbose=0)
cb_model.fit(X_cat_train, y_train, cat_features=cat_features)

cb_roc = roc_auc_score(y_test, cb_model.predict_proba(X_cat_test)[:, 1])
cb_f1 = f1_score(y_test, cb_model.predict(X_cat_test))

--- Обучение CatBoost ---


In [17]:
# ==========================================
# ШАГ 5. ИЗВЛЕЧЕНИЕ SHAP И ТОП-15 ПРИЗНАКОВ
# ==========================================
print("--- Расчет SHAP ---")
explainer = shap.TreeExplainer(cb_model)
shap_values = explainer.shap_values(X_cat_train)

shap_sum = np.abs(shap_values).mean(axis=0)
importance_df = pd.DataFrame({
    'feature': X_cat_train.columns,
    'importance': shap_sum
}).sort_values('importance', ascending=False)

top_15_features = importance_df.head(15)['feature'].tolist()

print("ТОП-15 ЧЕСТНЫХ ПРИЗНАКОВ ПО SHAP:")
for i, f in enumerate(top_15_features, 1):
    print(f"{i}. {f}")

--- Расчет SHAP ---
ТОП-15 ЧЕСТНЫХ ПРИЗНАКОВ ПО SHAP:
1. year
2. region_id
3. lat
4. vh_is_solo
5. viol_runaway
6. long
7. viol_pedestrian
8. light_cat
9. ppl_drv_female
10. near_administrative
11. near_residential
12. vh_mean_age
13. hour_cos
14. near_road_junctions
15. vh_age_gap


In [18]:
# ==========================================
# ШАГ 6. "ГРЯЗНАЯ" ЭКОНОМЕТРИКА (НА ВСЕХ ОСТАВШИХСЯ ПРИЗНАКАХ)
# ==========================================
scaler_base = StandardScaler()
X_logit_train_scaled = scaler_base.fit_transform(X_logit_train)
X_logit_test_scaled = scaler_base.transform(X_logit_test)

logit_base = LogisticRegression(max_iter=2000, random_state=42, class_weight='balanced')
logit_base.fit(X_logit_train_scaled, y_train)

lb_roc = roc_auc_score(y_test, logit_base.predict_proba(X_logit_test_scaled)[:, 1])
lb_f1 = f1_score(y_test, logit_base.predict(X_logit_test_scaled))

In [19]:
# ==========================================
# ШАГ 7. "УМНАЯ" ЭКОНОМЕТРИКА (ТОЛЬКО ТОП-15 ОТ SHAP)
# ==========================================
# Ищем колонки в X_logit, соответствующие топ-15 (с учетом OHE)
top_logit_cols = [c for c in X_logit_train.columns if any(c == f or c.startswith(f + '_') for f in top_15_features)]

X_train_smart = X_logit_train[top_logit_cols]
X_test_smart = X_logit_test[top_logit_cols]

scaler_smart = StandardScaler()
X_train_smart_scaled = scaler_smart.fit_transform(X_train_smart)
X_test_smart_scaled = scaler_smart.transform(X_test_smart)

logit_smart = LogisticRegression(max_iter=2000, random_state=42, class_weight='balanced')
logit_smart.fit(X_train_smart_scaled, y_train)

ls_roc = roc_auc_score(y_test, logit_smart.predict_proba(X_test_smart_scaled)[:, 1])
ls_f1 = f1_score(y_test, logit_smart.predict(X_test_smart_scaled))

In [20]:
# ==========================================
# ШАГ 8. ПРОВЕРКА VIF ДЛЯ УМНОГО ЛОГИТА
# ==========================================
print("--- Проверка VIF (Умный Логит) ---")
X_vif_clean = X_train_smart.astype(float)
X_vif_clean = X_vif_clean.loc[:, X_vif_clean.std() > 0]
X_vif = add_constant(X_vif_clean)

vifs = []
for i in range(X_vif.shape[1]):
    try:
        vifs.append(variance_inflation_factor(X_vif.values, i))
    except:
        vifs.append(np.inf)
        
vif_df = pd.DataFrame({'Feature': X_vif.columns, 'VIF': vifs})
print(vif_df[vif_df['Feature'] != 'const'].sort_values('VIF', ascending=False).head(5))

--- Проверка VIF (Умный Логит) ---
                Feature       VIF
13  light_cat_night_lit  2.312890
2              hour_cos  2.309925
6            vh_is_solo  2.068312
7            vh_age_gap  1.754598
10      viol_pedestrian  1.415494


In [21]:
# ==========================================
# ШАГ 9. ИНТЕРПРЕТАЦИЯ (ODDS RATIOS)
# ==========================================
print("--- ИНТЕРПРЕТАЦИЯ (Отношение шансов) ---")
coefs = logit_smart.coef_[0] / scaler_smart.scale_
odds_ratios = np.exp(coefs)

interpretation = pd.DataFrame({
    'Feature': top_logit_cols,
    'Odds_Ratio': odds_ratios
}).sort_values('Odds_Ratio', ascending=False)
print(interpretation)

--- ИНТЕРПРЕТАЦИЯ (Отношение шансов) ---
                 Feature  Odds_Ratio
11  light_cat_night_dark    3.134426
5             vh_is_solo    1.395664
9        viol_pedestrian    1.289479
1               hour_cos    1.214858
7            vh_mean_age    1.018199
6             vh_age_gap    1.002314
12   light_cat_night_lit    0.962091
0                   year    0.911864
4    near_administrative    0.885505
13    light_cat_twilight    0.867188
3    near_road_junctions    0.848829
8         ppl_drv_female    0.791715
2       near_residential    0.704057
10          viol_runaway    0.457557


In [22]:
# ==========================================
# ФИНАЛЬНЫЕ МЕТРИКИ
# ==========================================
print("==========================================")
print("ФИНАЛЬНЫЕ МЕТРИКИ (ТЕСТОВАЯ ВЫБОРКА):")
print(f"1. CatBoost (Честный, все признаки):   ROC-AUC = {cb_roc:.4f} | F1 = {cb_f1:.4f}")
print(f"2. Logit (Честный, все признаки):      ROC-AUC = {lb_roc:.4f} | F1 = {lb_f1:.4f}")
print(f"3. Logit (Честный, Топ-15 SHAP):       ROC-AUC = {ls_roc:.4f} | F1 = {ls_f1:.4f}")
print("==========================================")

ФИНАЛЬНЫЕ МЕТРИКИ (ТЕСТОВАЯ ВЫБОРКА):
1. CatBoost (Честный, все признаки):   ROC-AUC = 0.7550 | F1 = 0.5438
2. Logit (Честный, все признаки):      ROC-AUC = 0.6693 | F1 = 0.4757
3. Logit (Честный, Топ-15 SHAP):       ROC-AUC = 0.6490 | F1 = 0.4617


In [24]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from statsmodels.tools.tools import add_constant
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Предполагается, что X_logit_train_scaled, y_train, y_test и top_15_features (и дальше) у нас уже есть из прошлого кода.
# Если у тебя есть shap_importance DataFrame со всеми признаками, используй его.
# Допустим, all_shap_features - это список всех колонок X_cat_train, отсортированных по SHAP по убыванию.
all_shap_features = importance_df['feature'].tolist()

print("=========================================")
print("ЭКСПЕРИМЕНТ 1: LASSO РЕГУЛЯРИЗАЦИЯ (L1)")
print("=========================================")
# L1 регуляризация требует solver='liblinear' или 'saga'
logit_lasso = LogisticRegression(penalty='l1', solver='liblinear', C=0.1, max_iter=1000, random_state=42, class_weight='balanced')
logit_lasso.fit(X_logit_train_scaled, y_train)

# Считаем, сколько коэффициентов Lasso не занулил
non_zero_coefs = sum(logit_lasso.coef_[0] != 0)
lasso_roc = roc_auc_score(y_test, logit_lasso.predict_proba(X_logit_test_scaled)[:, 1])

print(f"ROC-AUC (Lasso на всех признаках): {lasso_roc:.4f}")
print(f"Lasso оставил признаков (не равных нулю): {non_zero_coefs} из {X_logit_train_scaled.shape[1]}")


print("\n=========================================")
print("ЭКСПЕРИМЕНТ 2: ПЕРЕБОР ЧИСЛА ПРИЗНАКОВ (SHAP)")
print("=========================================")

results = []

# Идем шагом в 5 признаков: от Топ-5 до Топ-30
for k in [5, 10, 15, 20, 25, 30]:
    # Берем Топ-K фичей от SHAP
    current_top = all_shap_features[:k]
    
    # Находим их в OHE-датасете
    top_cols = [c for c in X_logit_train.columns if any(c == f or c.startswith(f + '_') for f in current_top)]
    
    X_train_k = X_logit_train[top_cols]
    X_test_k = X_logit_test[top_cols]
    
    # 1. Проверяем максимальный VIF (Мультиколлинеарность)
    X_vif_clean = X_train_k.astype(float)
    X_vif_clean = X_vif_clean.loc[:, X_vif_clean.std() > 0]
    X_vif = add_constant(X_vif_clean)
    
    vifs = []
    for i in range(X_vif.shape[1]):
        try:
            vifs.append(variance_inflation_factor(X_vif.values, i))
        except:
            vifs.append(np.inf)
    
    # Берем максимальный VIF (исключая константу)
    max_vif = max(vifs[1:]) if len(vifs) > 1 else 1.0
    
    # 2. Обучаем стандартный Логит (без пенальти, чтобы проверить чистоту)
    scaler = StandardScaler()
    X_train_k_scaled = scaler.fit_transform(X_train_k)
    X_test_k_scaled = scaler.transform(X_test_k)
    
    logit_k = LogisticRegression(penalty=None, max_iter=1000, random_state=42, class_weight='balanced')
    # Для старых версий sklearn используй penalty='none'
    try:
        logit_k.fit(X_train_k_scaled, y_train)
    except ValueError:
        logit_k = LogisticRegression(penalty='none', max_iter=1000, random_state=42, class_weight='balanced')
        logit_k.fit(X_train_k_scaled, y_train)
        
    roc_k = roc_auc_score(y_test, logit_k.predict_proba(X_test_k_scaled)[:, 1])
    
    results.append({'Top_K': k, 'Features_Used': len(top_cols), 'Max_VIF': max_vif, 'ROC_AUC': roc_k})

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

ЭКСПЕРИМЕНТ 1: LASSO РЕГУЛЯРИЗАЦИЯ (L1)
ROC-AUC (Lasso на всех признаках): 0.6694
Lasso оставил признаков (не равных нулю): 59 из 63

ЭКСПЕРИМЕНТ 2: ПЕРЕБОР ЧИСЛА ПРИЗНАКОВ (SHAP)
 Top_K  Features_Used  Max_VIF  ROC_AUC
     5              3 1.074285 0.621873
    10              9 1.467140 0.633639
    15             14 2.312890 0.648955
    20             19 2.852643 0.657431
    25             24 8.960268 0.662254
    30             29 8.966910 0.664227
